The attached files are a collection of tweets labelled with sentiment in 3 categories:

sentiments = {
    "LABEL_0": "Bearish", 
    "LABEL_1": "Bullish", 
    "LABEL_2": "Neutral"
}

Train a LSTM network to with the training file. Validate the trained model with the valid file. Comment what you are doing in each part of your code. As the better the code, comments and result validation as the better the grade.

In [1]:
!pip install transformers datasets evaluate accelerate

  Using cached transformers-4.50.0-py3-none-any.whl.metadata (39 kB)
  Using cached huggingface_hub-0.29.3-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.21.1-cp39-abi3-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached safetensors-0.5.3-cp38-abi3-macosx_11_0_arm64.whl.metadata (3.8 kB)
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
Using cached transformers-4.50.0-py3-none-any.whl (10.2 MB)
Using cached huggingface_hub-0.29.3-py3-none-any.whl (468 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 592.6 kB/s eta 0:00:0000:0100:02
Using cached requests-2.32.3-py3-none-any.whl (64 kB)
Using cached safetensors-0.5.3-cp38-abi3-macosx_11_0_arm64.whl (418 kB)
Using cached tokenizers-0.21.1-cp39-abi3-macosx_11_0_arm64.whl (2.7 MB)
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
  Attempting uninstall: fsspec
    Found existing inst

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

ModuleNotFoundError: No module named 'urllib3.packages.six.moves'

In [ ]:
# Load the dataset, each tweet of the dataset have a prompt that is the tweet text and the label that is the type of the email (0,1,2)
file_path_train = "/Users/bernardoquindimil/Code/Berniquindimil/NLP_Digital_Portfolio/S09/sent_train.csv"
df_train = pd.read_csv(file_path_train)

file_path_valid = "/Users/bernardoquindimil/Code/Berniquindimil/NLP_Digital_Portfolio/S09/sent_valid.csv"
df_test = pd.read_csv(file_path_valid)

In [8]:
# The dataset has 'text' and 'label' columns
texts_train = df_train['text'].astype(str).values  # Convert to string in case of NaN
labels_train = df_train['label'].values

# Encode labels
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels_train)

## Tokenization

In [9]:
max_words = 10000  # Tamaño máximo del vocabulario
max_len = 200  # Longitud máxima de la secuencia
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts_train)
sequences = tokenizer.texts_to_sequences(texts_train)
X = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42)

## LSTM model

In [ ]:
# Create the LSTM model
model_lstm = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(64, return_sequences=True),
    Dropout(0.5),
    LSTM(32),
    Dense(32, activation='relu'),
    Dense(1, activation='softmax')  # Softmax for multiple class
])

/Users/bernardoquindimil/miniconda3/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


The RNNs has the disadvantage that store a lot of words for proving a single word and some of those words are not related with the single word. LSTM store the words that are related creating a context without non-related words. This characteristic allows LSTM to avoid irrelevant information and store words far away.

In [ ]:
model_lstm.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])  # Binary cossentropy because in most of cases, is bettet than one. Binary crossentropy means that not only takes in account the order from left to right.

In [ ]:
# Training the LSTM model
model_lstm.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5, batch_size=32) # Train with the sent_train.csv data doing a partition and validate with the sent_valid.csv

Epoch 1/15
239/239 ━━━━━━━━━━━━━━━━━━━━ 26s 108ms/step - accuracy: 0.2022 - loss: -807.2305 - val_accuracy: 0.2048 - val_loss: -1014.7200
Epoch 2/15
239/239 ━━━━━━━━━━━━━━━━━━━━ 25s 103ms/step - accuracy: 0.1963 - loss: -1111.3352 - val_accuracy: 0.2048 - val_loss: -1345.7598
Epoch 3/15
239/239 ━━━━━━━━━━━━━━━━━━━━ 25s 103ms/step - accuracy: 0.2071 - loss: -1434.1351 - val_accuracy: 0.2048 - val_loss: -1716.3746
Epoch 4/15
239/239 ━━━━━━━━━━━━━━━━━━━━ 25s 105ms/step - accuracy: 0.1962 - loss: -1817.3123 - val_accuracy: 0.2048 - val_loss: -2122.1418
Epoch 5/15
239/239 ━━━━━━━━━━━━━━━━━━━━ 27s 112ms/step - accuracy: 0.1996 - loss: -2247.7590 - val_accuracy: 0.2048 - val_loss: -2562.8535
Epoch 6/15
239/239 ━━━━━━━━━━━━━━━━━━━━ 25s 106ms/step - accuracy: 0.2024 - loss: -2728.6545 - val_accuracy: 0.2048 - val_loss: -3038.4324
Epoch 7/15
239/239 ━━━━━━━━━━━━━━━━━━━━ 26s 108ms/step - accuracy: 0.1987 - loss: -3215.5793 - val_accuracy: 0.2048 - val_loss: -3543.6526
Epoch 8/15
167/239 ━━━━━━━━━

KeyboardInterrupt: 

In [14]:
# Evaluate LSTM model
loss_lstm, accuracy_lstm = model_lstm.evaluate(X_test, y_test)
print(f"LSTM Test Accuracy: {accuracy_lstm:.4f}")

60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.2247 - loss: -688.7380
LSTM Test Accuracy: 0.2048


I have an accuracy of 0.2048 that is not good because I have done very few eppoch. A solution is training with more eppochs.